#Foundation questions (1-10)


# How does select differ from withColumn?

Select : is only use to select the required columns in dataset so that it is easy to work with the selected columns instead of confusing with entire dataset
mainly used to choose,rename or create expression for columns and return only the columns specified in the result

WithColumn : is used to create a new columns or modify/replace the existing columns as per the business requirement


#How do you refer to a column using functions.col?


from pyspark.sql import functions as F
#Syntax
F.col("column_name")

functions.col() is used to reference a DataFrame column by its name . it is commonly used when performing transformations,filtering or creating new columns

#What does cast change about a column?

The cast is a Type convert a column from one data type to another  based on a business column

example : in Dataset if the date column is in string so we can use cast keyword to change the date column from string to date

df=df.withColumn("date",F.col("date").cast("date"))

#Why must CSV quantity be typed before arithmetic?

CSV data is often read as string when the schema is not explicitly defined before performing airthemetic on a quantity column, we need to cast it to a numeric data type such as int,long and float because airthmetic cannot reliably be performed on a string values


# How do filter and where relate?

Filter() and where() are equivalent in Pyspark.Both are used to filter rows from a dataFrame based on a given condition .where() is essentially an alias for filter()


# How do you combine column conditions using & and |?

df.filter((col("salary")> 100) & (col("salary") < 1000))

df.filter((col("salary")> 100) | (col("salary") < 1000))

df.filter((col("salary")> 100) ~ (col("salary") < 1000))

In PySpark, we use & to combine conditions with AND logic and | to combine conditions with OR logic. Each condition should be enclosed in parentheses.






#Why are parentheses needed around combined comparisons?

 Parantheses are needed around combined comparision primarly because of operator precendence = the built in mathematical rules that dectate the order in which a computer program executes operations

so the flow : PEMDAS, that is, Parenthesis, Exponentiation, Division, Multiplication, Addition, Subtraction. Note that the precedence order of Division and Multiplication is the same. Likewise, the order of Addition and Subtraction is also the same.


#What does when(...).otherwise(...) express?

when(...).otherwise(...) in PySpark is similar to an if-else statement. The when() condition is evaluated first. If the condition is true, its corresponding value is returned. If the condition is false, the otherwise() value is returned.

#How does isNull differ from comparing a column to None?

isNull(): is a function that scans the data rows and is used to check whether the existing column has null values and returns the column with Null

None : is the value which is used when the data got missing so that in backend it treats it as a null value

#How are line gross and line net defined in this course?

• Line Gross (line_gross): The calculated total for an individual row/record before any reductions (like promotions or discounts) or additions (like taxes).
line_gross=text{quantity} *{unit_price}

• Line Net (line_net): The final calculated value for that individual row after applying all deductions.
line_net=line_gross-{line_discount}


#Hands-on questions (11-20)


In [1]:
pip install pyspark

In [2]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("filtering").getOrCreate()
from pyspark.sql import functions as F

In [3]:
customers=[(101,"Ramya","ramya@gmail.com","Hyderabad","2024-01-15",True),
            (102,"Anil",None,"Bangalore","2024-02-20",True),
            (103,"Sita","sita@gmail.com","Chennai","2024-03-10",False),
            (104,"Ravi","ravi@gmail.com","Mumbai","2024-04-05",True),
            (101,"Priya","priya@gmail.com","Delhi","2024-05-12",True),
            (105,"Kiran","kiran@gmail.com","Pune","2024-06-18",False),
            (104,"Meena","meena@gmail.com","Kolkata","2024-07-22",True),
            (108,"Arjun","arjun@gmail.com","Hyderabad","2024-08-09",True),
            (109,"Divya","divya@gmail.com","Bangalore","2024-09-14",True),
            (103,"Varun","varun@gmail.com",None,"2024-10-25",False)
]

schema=StructType([
    StructField("customer_id",IntegerType(),False),
    StructField("first_name",StringType(),False),
    StructField("email",StringType(),True),
    StructField("city",StringType(),True),
    StructField("registration_date",StringType(),False),
    StructField("is_active",BooleanType())
])

df_customers=spark.createDataFrame(customers,schema)
df_customers=df_customers.withColumn("registration_date",to_date(col("registration_date"),"yyyy-MM-dd"))
df_customers.printSchema()
df_customers.show(truncate=False)



root
 |-- customer_id: integer (nullable = false)
 |-- first_name: string (nullable = false)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- registration_date: date (nullable = false)
 |-- is_active: boolean (nullable = true)

+-----------+----------+---------------+---------+-----------------+---------+
|customer_id|first_name|email          |city     |registration_date|is_active|
+-----------+----------+---------------+---------+-----------------+---------+
|101        |Ramya     |ramya@gmail.com|Hyderabad|2024-01-15       |true     |
|102        |Anil      |NULL           |Bangalore|2024-02-20       |true     |
|103        |Sita      |sita@gmail.com |Chennai  |2024-03-10       |false    |
|104        |Ravi      |ravi@gmail.com |Mumbai   |2024-04-05       |true     |
|101        |Priya     |priya@gmail.com|Delhi    |2024-05-12       |true     |
|105        |Kiran     |kiran@gmail.com|Pune     |2024-06-18       |false    |
|104        |Meena     |meena@gm

In [4]:
from decimal import Decimal

In [5]:
# Define a complete order_items schema with decimal prices and discounts.
order_items = [
    (1001, 5001, 201, "Laptop Stand",        2, Decimal("499.99"),  Decimal("10.50")),
    (1002, 5001, 202, "Wireless Keyboard",  None, Decimal("1299.75"), Decimal("50.25")),
    (1003, 5002, 203, "USB Cable",          5, Decimal("99.99"),   Decimal("5.00")),
    (1004, 5002, 204, "Wireless Mouse",     3, Decimal("250.50"),  Decimal("15.75")),
    (1005, 5003, 205, "Webcam",             1, Decimal("999.99"),  Decimal("0.00")),
    (1006, 5003, 201, "Laptop Stand",       4, Decimal("499.99"),  Decimal("25.50")),
    (1007, 5004, 206, "Monitor",            2, Decimal("750.25"),  Decimal("20.00")),
    (1008, 5005, 207, "Mechanical Keyboard",1, Decimal("1599.90"), Decimal("100.50")),
    (1009, 5005, 208, "Mouse Pad",          6, Decimal("75.25"),   Decimal("5.25")),
    (1010, 5006, 209, "USB Hub",            2, Decimal("349.99"),  None),
    (1011, 5007, 210, "Headphones",         3, Decimal("450.75"),  Decimal("30.25")),
    (1012, 5008, 203, "USB Cable",          2, Decimal("99.99"),   Decimal("10.00")),
    (1013, 5009, 211, "4K Monitor",         1, Decimal("2499.99"), Decimal("150.75")),
    (1014, 5010, 212, "Webcam Cover",       5, Decimal("125.50"),  Decimal("12.50")),
    (1015, 5010, 201, "Laptop Stand",       None, Decimal("499.99"),  Decimal("50.00"))
]

schema=StructType([
    StructField("order_item_id",IntegerType(),False),
    StructField("order_id",IntegerType(),True),
    StructField("product_id",IntegerType(),True),
    StructField("product_name",StringType(),True),
    StructField("quantity",IntegerType(),True),
    StructField("unit_price",DecimalType(10,2),True),
    StructField("discount",DecimalType(10,2),True)
])
df_order_items=spark.createDataFrame(order_items,schema)
df_order_items.printSchema()
df_order_items.show(truncate=False)

root
 |-- order_item_id: integer (nullable = false)
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- discount: decimal(10,2) (nullable = true)

+-------------+--------+----------+-------------------+--------+----------+--------+
|order_item_id|order_id|product_id|product_name       |quantity|unit_price|discount|
+-------------+--------+----------+-------------------+--------+----------+--------+
|1001         |5001    |201       |Laptop Stand       |2       |499.99    |10.50   |
|1002         |5001    |202       |Wireless Keyboard  |NULL    |1299.75   |50.25   |
|1003         |5002    |203       |USB Cable          |5       |99.99     |5.00    |
|1004         |5002    |204       |Wireless Mouse     |3       |250.50    |15.75   |
|1005         |5003    |205       |Webcam             |1       |999.99    |0.00    |
|1

In [6]:
# Calculate net value for quantity 2, price 500, and discount 0.10.
values=[(2,500,0.10)]
schema=StructType([
    StructField("quantity",IntegerType(),True),
    StructField("price",IntegerType(),True),
    StructField("discount",DoubleType(),False)
])
df=spark.createDataFrame(values,schema)
df_alculate=df.withColumn("net_value",(col("quantity")*col("price"))*(1-col("discount")))
df_alculate.show()
df_alculate.printSchema()

+--------+-----+--------+---------+
|quantity|price|discount|net_value|
+--------+-----+--------+---------+
|       2|  500|     0.1|    900.0|
+--------+-----+--------+---------+

root
 |-- quantity: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- discount: double (nullable = false)
 |-- net_value: double (nullable = true)



In [7]:
# Create decimal quantity-price-discount calculations for all line items.
df_order_items = df_order_items.withColumn("net_amount",(col("quantity")*col("unit_price")) * (1-(col("discount")/100)))
df_order_items.show()

+-------------+--------+----------+-------------------+--------+----------+--------+--------------+
|order_item_id|order_id|product_id|       product_name|quantity|unit_price|discount|    net_amount|
+-------------+--------+----------+-------------------+--------+----------+--------+--------------+
|         1001|    5001|       201|       Laptop Stand|       2|    499.99|   10.50|  894.98210000|
|         1002|    5001|       202|  Wireless Keyboard|    NULL|   1299.75|   50.25|          NULL|
|         1003|    5002|       203|          USB Cable|       5|     99.99|    5.00|  474.95250000|
|         1004|    5002|       204|     Wireless Mouse|       3|    250.50|   15.75|  633.13875000|
|         1005|    5003|       205|             Webcam|       1|    999.99|    0.00|  999.99000000|
|         1006|    5003|       201|       Laptop Stand|       4|    499.99|   25.50| 1489.97020000|
|         1007|    5004|       206|            Monitor|       2|    750.25|   20.00| 1200.40000000|


In [8]:
# Round line net to two decimal places using the course rounding rule.
df_order_items=df_order_items.withColumn("net_amount",round(col("net_amount"),2))
df_order_items.show()

+-------------+--------+----------+-------------------+--------+----------+--------+----------+
|order_item_id|order_id|product_id|       product_name|quantity|unit_price|discount|net_amount|
+-------------+--------+----------+-------------------+--------+----------+--------+----------+
|         1001|    5001|       201|       Laptop Stand|       2|    499.99|   10.50|    894.98|
|         1002|    5001|       202|  Wireless Keyboard|    NULL|   1299.75|   50.25|      NULL|
|         1003|    5002|       203|          USB Cable|       5|     99.99|    5.00|    474.95|
|         1004|    5002|       204|     Wireless Mouse|       3|    250.50|   15.75|    633.14|
|         1005|    5003|       205|             Webcam|       1|    999.99|    0.00|    999.99|
|         1006|    5003|       201|       Laptop Stand|       4|    499.99|   25.50|   1489.97|
|         1007|    5004|       206|            Monitor|       2|    750.25|   20.00|   1200.40|
|         1008|    5005|       207|Mecha

In [9]:
# Filter positive quantities with discount_pct between zero and one inclusive.
df_order_items_positive=df_order_items.withColumn("discount_pct",col("discount")/100)\
                             .filter((col("discount_pct") > 0) & (col("discount_pct") <=1))
df_order_items_positive.show()


+-------------+--------+----------+-----------------+--------+----------+--------+----------+------------+
|order_item_id|order_id|product_id|     product_name|quantity|unit_price|discount|net_amount|discount_pct|
+-------------+--------+----------+-----------------+--------+----------+--------+----------+------------+
|         1001|    5001|       201|     Laptop Stand|       2|    499.99|   10.50|    894.98|    0.105000|
|         1002|    5001|       202|Wireless Keyboard|    NULL|   1299.75|   50.25|      NULL|    0.502500|
|         1003|    5002|       203|        USB Cable|       5|     99.99|    5.00|    474.95|    0.050000|
|         1004|    5002|       204|   Wireless Mouse|       3|    250.50|   15.75|    633.14|    0.157500|
|         1006|    5003|       201|     Laptop Stand|       4|    499.99|   25.50|   1489.97|    0.255000|
|         1007|    5004|       206|          Monitor|       2|    750.25|   20.00|   1200.40|    0.200000|
|         1009|    5005|       208|  

In [10]:
#Classify amounts below 500, below 2000, and all remaining amounts.
df_order_items_category=df_order_items.withColumn("net_amount_category",
                                                  F.when(F.col("net_amount")<500,"Below 500")
                                                  .when(F.col("net_amount")<2000,"Below 2000")
                                                  .when(F.col("net_amount").isNull(),"No amount")
                                                  .otherwise ("2000 and above"))
df_order_items_category.show()


+-------------+--------+----------+-------------------+--------+----------+--------+----------+-------------------+
|order_item_id|order_id|product_id|       product_name|quantity|unit_price|discount|net_amount|net_amount_category|
+-------------+--------+----------+-------------------+--------+----------+--------+----------+-------------------+
|         1001|    5001|       201|       Laptop Stand|       2|    499.99|   10.50|    894.98|         Below 2000|
|         1002|    5001|       202|  Wireless Keyboard|    NULL|   1299.75|   50.25|      NULL|          No amount|
|         1003|    5002|       203|          USB Cable|       5|     99.99|    5.00|    474.95|          Below 500|
|         1004|    5002|       204|     Wireless Mouse|       3|    250.50|   15.75|    633.14|         Below 2000|
|         1005|    5003|       205|             Webcam|       1|    999.99|    0.00|    999.99|         Below 2000|
|         1006|    5003|       201|       Laptop Stand|       4|    499.

In [11]:
# Find lines with net amount greater than 2000.
df_order_items_lines=df_order_items.filter(F.col("net_amount")>2000)
df_order_items_lines.show()

+-------------+--------+----------+------------+--------+----------+--------+----------+
|order_item_id|order_id|product_id|product_name|quantity|unit_price|discount|net_amount|
+-------------+--------+----------+------------+--------+----------+--------+----------+
+-------------+--------+----------+------------+--------+----------+--------+----------+



In [12]:
df_order_items.show()

+-------------+--------+----------+-------------------+--------+----------+--------+----------+
|order_item_id|order_id|product_id|       product_name|quantity|unit_price|discount|net_amount|
+-------------+--------+----------+-------------------+--------+----------+--------+----------+
|         1001|    5001|       201|       Laptop Stand|       2|    499.99|   10.50|    894.98|
|         1002|    5001|       202|  Wireless Keyboard|    NULL|   1299.75|   50.25|      NULL|
|         1003|    5002|       203|          USB Cable|       5|     99.99|    5.00|    474.95|
|         1004|    5002|       204|     Wireless Mouse|       3|    250.50|   15.75|    633.14|
|         1005|    5003|       205|             Webcam|       1|    999.99|    0.00|    999.99|
|         1006|    5003|       201|       Laptop Stand|       4|    499.99|   25.50|   1489.97|
|         1007|    5004|       206|            Monitor|       2|    750.25|   20.00|   1200.40|
|         1008|    5005|       207|Mecha

In [13]:
# Select only line_id and computed amounts without modifying raw columns.
df_order_items.select(F.col("order_id"),F.col("net_amount")).show()

+--------+----------+
|order_id|net_amount|
+--------+----------+
|    5001|    894.98|
|    5001|      NULL|
|    5002|    474.95|
|    5002|    633.14|
|    5003|    999.99|
|    5003|   1489.97|
|    5004|   1200.40|
|    5005|     -8.00|
|    5005|    427.80|
|    5006|      NULL|
|    5007|    943.19|
|    5008|    179.98|
|    5009|  -1268.74|
|    5010|    549.06|
|    5010|      NULL|
+--------+----------+



In [14]:
# Create fixtures for discounts 0, 1, -0.1, and 1.1.
from decimal import Decimal

discount_fixtures = [
    (1, 1001, 2, Decimal("100.00"), Decimal("0.0")),
    (2, 1002, 3, Decimal("200.00"), Decimal("1.0")),
    (3, 1003, 1, Decimal("500.00"), Decimal("-0.1")),
    (4, 1004, 2, Decimal("300.00"), Decimal("1.1"))
]
schema=StructType([
    StructField("line_id",IntegerType(),True),
    StructField("order_id",IntegerType(),True),
    StructField("quantity",IntegerType(),True),
    StructField("unit_price",DecimalType(10,2),True),
    StructField("discount_pct",DecimalType(10,2),True)
])
df_fixtures=spark.createDataFrame(discount_fixtures,schema)
df_fixture_filter=df_fixtures.filter((F.col("discount_pct")<=0) & (F.col("discount_pct")<=1))
df_fixture_filter.show()


+-------+--------+--------+----------+------------+
|line_id|order_id|quantity|unit_price|discount_pct|
+-------+--------+--------+----------+------------+
|      1|    1001|       2|    100.00|        0.00|
|      3|    1003|       1|    500.00|       -0.10|
+-------+--------+--------+----------+------------+



In [15]:
df_order_items.show()

+-------------+--------+----------+-------------------+--------+----------+--------+----------+
|order_item_id|order_id|product_id|       product_name|quantity|unit_price|discount|net_amount|
+-------------+--------+----------+-------------------+--------+----------+--------+----------+
|         1001|    5001|       201|       Laptop Stand|       2|    499.99|   10.50|    894.98|
|         1002|    5001|       202|  Wireless Keyboard|    NULL|   1299.75|   50.25|      NULL|
|         1003|    5002|       203|          USB Cable|       5|     99.99|    5.00|    474.95|
|         1004|    5002|       204|     Wireless Mouse|       3|    250.50|   15.75|    633.14|
|         1005|    5003|       205|             Webcam|       1|    999.99|    0.00|    999.99|
|         1006|    5003|       201|       Laptop Stand|       4|    499.99|   25.50|   1489.97|
|         1007|    5004|       206|            Monitor|       2|    750.25|   20.00|   1200.40|
|         1008|    5005|       207|Mecha

In [16]:
# Compare filtering null quantities with isNull and a numeric comparison.
df_order_null_numeric=df_order_items.filter((F.col("quantity").isNull()) | (F.col("quantity")==0))
df_order_null_numeric.show()

+-------------+--------+----------+-----------------+--------+----------+--------+----------+
|order_item_id|order_id|product_id|     product_name|quantity|unit_price|discount|net_amount|
+-------------+--------+----------+-----------------+--------+----------+--------+----------+
|         1002|    5001|       202|Wireless Keyboard|    NULL|   1299.75|   50.25|      NULL|
|         1015|    5010|       201|     Laptop Stand|    NULL|    499.99|   50.00|      NULL|
+-------------+--------+----------+-----------------+--------+----------+--------+----------+



In [17]:
# Create a named reusable expression for net line amount.
net_calculate_amount=(col("quantity") * col("unit_price")) * (1-(col("discount")/100))

#Advanced challenges (21-30)


In [18]:
# Compare rounding each line before aggregation with rounding only the final total.
df_order_items.show()
df_order_round=df_order_items.select("order_id","net_amount").groupBy("order_id").agg(round(sum("net_amount"),2).alias("total_round_final"))
df_order_round.show()

+-------------+--------+----------+-------------------+--------+----------+--------+----------+
|order_item_id|order_id|product_id|       product_name|quantity|unit_price|discount|net_amount|
+-------------+--------+----------+-------------------+--------+----------+--------+----------+
|         1001|    5001|       201|       Laptop Stand|       2|    499.99|   10.50|    894.98|
|         1002|    5001|       202|  Wireless Keyboard|    NULL|   1299.75|   50.25|      NULL|
|         1003|    5002|       203|          USB Cable|       5|     99.99|    5.00|    474.95|
|         1004|    5002|       204|     Wireless Mouse|       3|    250.50|   15.75|    633.14|
|         1005|    5003|       205|             Webcam|       1|    999.99|    0.00|    999.99|
|         1006|    5003|       201|       Laptop Stand|       4|    499.99|   25.50|   1489.97|
|         1007|    5004|       206|            Monitor|       2|    750.25|   20.00|   1200.40|
|         1008|    5005|       207|Mecha

In [19]:
# Demonstrate half-up rounding and explain how bround differs on a tie.
df_rounding = spark.createDataFrame([(10.125,),(10.135,),(10.124,)],["amount"])
df_rounding=df_rounding.select(("amount"),F.round("amount",2).alias("half_up"),F.bround("amount",2).alias("half_even"))
df_rounding.show()


+------+-------+---------+
|amount|half_up|half_even|
+------+-------+---------+
|10.125|  10.13|    10.12|
|10.135|  10.14|    10.14|
|10.124|  10.12|    10.12|
+------+-------+---------+



In [20]:
# Create an overflow fixture and document behavior with ANSI mode enabled and disabled.
df_overflow=spark.createDataFrame([(214783647,),(214783646,)],["amounts"])
spark.conf.set("spark.sql.ansi.enabled","true")
df_overflow = df_overflow .select(F.expr("cast(amounts as INT)+1").alias("result"))
df_overflow.show()

+---------+
|   result|
+---------+
|214783648|
|214783647|
+---------+



#Precision = total number of digits → 5

<!-- Scale = number of digits after the decimal point → 2

So:

123.45 → Precision = 5, Scale = 2
 -->


In [21]:
# Explain the precision and scale of intermediate decimal multiplication.
data=[
    (123.45,10.00),
    (50.25,3.00)
]
df_multiplication=spark.createDataFrame(data,["amount","quantity"])
df_multiplication.show()
df_multiplication = df_multiplication.withColumn("total", F.col("amount")*F.col("quantity"))
df_multiplication.show()

+------+--------+
|amount|quantity|
+------+--------+
|123.45|    10.0|
| 50.25|     3.0|
+------+--------+

+------+--------+------+
|amount|quantity| total|
+------+--------+------+
|123.45|    10.0|1234.5|
| 50.25|     3.0|150.75|
+------+--------+------+



In [22]:
df_order_items.show()

+-------------+--------+----------+-------------------+--------+----------+--------+----------+
|order_item_id|order_id|product_id|       product_name|quantity|unit_price|discount|net_amount|
+-------------+--------+----------+-------------------+--------+----------+--------+----------+
|         1001|    5001|       201|       Laptop Stand|       2|    499.99|   10.50|    894.98|
|         1002|    5001|       202|  Wireless Keyboard|    NULL|   1299.75|   50.25|      NULL|
|         1003|    5002|       203|          USB Cable|       5|     99.99|    5.00|    474.95|
|         1004|    5002|       204|     Wireless Mouse|       3|    250.50|   15.75|    633.14|
|         1005|    5003|       205|             Webcam|       1|    999.99|    0.00|    999.99|
|         1006|    5003|       201|       Laptop Stand|       4|    499.99|   25.50|   1489.97|
|         1007|    5004|       206|            Monitor|       2|    750.25|   20.00|   1200.40|
|         1008|    5005|       207|Mecha

In [24]:
# Handle null price and discount explicitly without silently treating them as zero.
df_order_items_handle=df_order_items.withColumn("final_price",
                                                when((col("quantity").isNotNull()) & (col("discount").isNotNull()),
                                                col("unit_price")-col("discount"))
                                                .otherwise(lit(None))

                                                )
df_order_items_handle.show()


+-------------+--------+----------+-------------------+--------+----------+--------+----------+-----------+
|order_item_id|order_id|product_id|       product_name|quantity|unit_price|discount|net_amount|final_price|
+-------------+--------+----------+-------------------+--------+----------+--------+----------+-----------+
|         1001|    5001|       201|       Laptop Stand|       2|    499.99|   10.50|    894.98|     489.49|
|         1002|    5001|       202|  Wireless Keyboard|    NULL|   1299.75|   50.25|      NULL|       NULL|
|         1003|    5002|       203|          USB Cable|       5|     99.99|    5.00|    474.95|      94.99|
|         1004|    5002|       204|     Wireless Mouse|       3|    250.50|   15.75|    633.14|     234.75|
|         1005|    5003|       205|             Webcam|       1|    999.99|    0.00|    999.99|     999.99|
|         1006|    5003|       201|       Laptop Stand|       4|    499.99|   25.50|   1489.97|     474.49|
|         1007|    5004|    

In [28]:
# Test classification at 499.99, 500.00, 1999.99, and 2000.00.
df_order_items_classify=df_order_items.withColumn("Classification",
                                                  when(col("net_amount")<499.99,"Just below 500")
                                                  .when(col("net_amount")==500, "Exactly 500" )
                                                  .when(col("net_amount") < 1999.99,"Just below 2000")
                                                  .when((col("net_amount").isNull()),"Not assigned")
                                                  .otherwise("Exactly 2000")
                                                  )
df_order_items_classify.show()

+-------------+--------+----------+-------------------+--------+----------+--------+----------+---------------+
|order_item_id|order_id|product_id|       product_name|quantity|unit_price|discount|net_amount| Classification|
+-------------+--------+----------+-------------------+--------+----------+--------+----------+---------------+
|         1001|    5001|       201|       Laptop Stand|       2|    499.99|   10.50|    894.98|Just below 2000|
|         1002|    5001|       202|  Wireless Keyboard|    NULL|   1299.75|   50.25|      NULL|   Not assigned|
|         1003|    5002|       203|          USB Cable|       5|     99.99|    5.00|    474.95| Just below 500|
|         1004|    5002|       204|     Wireless Mouse|       3|    250.50|   15.75|    633.14|Just below 2000|
|         1005|    5003|       205|             Webcam|       1|    999.99|    0.00|    999.99|Just below 2000|
|         1006|    5003|       201|       Laptop Stand|       4|    499.99|   25.50|   1489.97|Just belo

## Explain why a negative quantity must not become a negative sale in this dataset.

A negative quantity should not automatically produce a negative sale because quantity represents the number of units sold . In a normal sales dataset, a negative quantity is generally invalid or indicate bad return / correction . Therefore , I would validate the quantity before calculating sales. if returns are not part of the dataset , I wwould trat negative quantities as invalid rather than allowing them to create negative sales

In [54]:
# Compare built-in column expressions with a Python loop over collected rows.
df_order_items_sale=df_order_items.withColumn("sale",col("unit_price")*col("quantity"))
df_order_items_sale.show()
rows=df_order_items.collect()
rows
for row in rows:
  if row["quantity"] is not None and row["unit_price"] is not None:
    sale=row["unit_price"] * row["quantity"]
    print(sale)


+-------------+--------+----------+-------------------+--------+----------+--------+----------+-------+
|order_item_id|order_id|product_id|       product_name|quantity|unit_price|discount|net_amount|   sale|
+-------------+--------+----------+-------------------+--------+----------+--------+----------+-------+
|         1001|    5001|       201|       Laptop Stand|       2|    499.99|   10.50|    894.98| 999.98|
|         1002|    5001|       202|  Wireless Keyboard|    NULL|   1299.75|   50.25|      NULL|   NULL|
|         1003|    5002|       203|          USB Cable|       5|     99.99|    5.00|    474.95| 499.95|
|         1004|    5002|       204|     Wireless Mouse|       3|    250.50|   15.75|    633.14| 751.50|
|         1005|    5003|       205|             Webcam|       1|    999.99|    0.00|    999.99| 999.99|
|         1006|    5003|       201|       Laptop Stand|       4|    499.99|   25.50|   1489.97|1999.96|
|         1007|    5004|       206|            Monitor|       2|

In [56]:
# Inspect whether chained projections are simplified in the optimized plan.
df_test=df_order_items.select("order_item_id","unit_price","quantity")\
                      .select("order_item_id","unit_price")
print(df_test.show())
print(df_test.explain(True))

+-------------+----------+
|order_item_id|unit_price|
+-------------+----------+
|         1001|    499.99|
|         1002|   1299.75|
|         1003|     99.99|
|         1004|    250.50|
|         1005|    999.99|
|         1006|    499.99|
|         1007|    750.25|
|         1008|   1599.90|
|         1009|     75.25|
|         1010|    349.99|
|         1011|    450.75|
|         1012|     99.99|
|         1013|   2499.99|
|         1014|    125.50|
|         1015|    499.99|
+-------------+----------+

None
== Parsed Logical Plan ==
'Project ['order_item_id, 'unit_price]
+- Project [order_item_id#26, unit_price#31, quantity#30]
   +- Project [order_item_id#26, order_id#27, product_id#28, product_name#29, quantity#30, unit_price#31, discount#32, round(net_amount#72, 2) AS net_amount#98]
      +- Project [order_item_id#26, order_id#27, product_id#28, product_name#29, quantity#30, unit_price#31, discount#32, ((cast(quantity#30 as decimal(10,0)) * unit_price#31) * (cast(1 as decimal(

In [61]:
# Write a calculation function and expected-output checks for all boundary cases.
print(net_calculate_amount)
def calculate_net_amount():
  return col("unit_price") * col("quantity")- col("discount")
df_test1=df_order_items.withColumn("calculated_net_amount",calculate_net_amount())
df_test1.select("order_item_id","unit_price","quantity","discount","calculated_net_amount").show()

Column<'*(*(quantity, unit_price), -(1, /(discount, 100)))'>
+-------------+----------+--------+--------+---------------------+
|order_item_id|unit_price|quantity|discount|calculated_net_amount|
+-------------+----------+--------+--------+---------------------+
|         1001|    499.99|       2|   10.50|               989.48|
|         1002|   1299.75|    NULL|   50.25|                 NULL|
|         1003|     99.99|       5|    5.00|               494.95|
|         1004|    250.50|       3|   15.75|               735.75|
|         1005|    999.99|       1|    0.00|               999.99|
|         1006|    499.99|       4|   25.50|              1974.46|
|         1007|    750.25|       2|   20.00|              1480.50|
|         1008|   1599.90|       1|  100.50|              1499.40|
|         1009|     75.25|       6|    5.25|               446.25|
|         1010|    349.99|       2|    NULL|                 NULL|
|         1011|    450.75|       3|   30.25|              1322.00|
|